In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("20-scd-patterns")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]

# ── Section 1 ── SCD Type 1: Overwrite current state (no history kept)
# employees table IS the Type 1 SCD table (current state only)
# "Fix" emp 22 NULL email and emp 19 salary=0.0
emp_type1 = emp.withColumn("email",
    F.when((F.col("emp_id") == 22) & F.col("email").isNull(), F.lit("v.lee@corp.com"))
     .otherwise(F.col("email"))
).withColumn("salary",
    F.when((F.col("emp_id") == 19) & (F.col("salary") == 0.0), F.lit(115000.0))
     .otherwise(F.col("salary"))
)
# Verify the fixes
emp_type1.filter(F.col("emp_id").isin(19, 22)).select("emp_id", "first_name", "email", "salary").show()


# ── Section 2 ── SCD Type 1 MERGE pattern (simulate upsert from staging)
staging = spark.createDataFrame([
    (22, "Victor", "Lee", "v.lee@corp.com",  95000.0, "Active"),   # fix NULL email
    (99, "New",    "Hire", "new@corp.com",   80000.0, "Active"),   # new employee
], ["emp_id", "first_name", "last_name", "email", "salary", "status"])

# MATCH: update existing rows; NOT MATCH: insert new rows
updated = emp.join(
    staging.select("emp_id", "email", "salary")
           .withColumnRenamed("email", "new_email")
           .withColumnRenamed("salary", "new_salary"),
    "emp_id", "left"
).withColumn("email",  F.coalesce(F.col("new_email"),  F.col("email"))) \
 .withColumn("salary", F.coalesce(F.col("new_salary"), F.col("salary"))) \
 .drop("new_email", "new_salary")

new_rows = staging.join(emp.select("emp_id"), "emp_id", "left_anti") \
                  .select(*emp.columns)

type1_result = updated.union(new_rows.select(*updated.columns))
print(f"After Type 1 merge: {type1_result.count()} rows")


# ── Section 3 ── SCD Type 2: salary_history IS the Type 2 table
# salary_before=None = initial hire; valid_to = LEAD of next effective_date
w_scd2 = Window.partitionBy("emp_id").orderBy("effective_date")
sal_scd2 = sal.withColumn("valid_to", F.lead("effective_date", 1).over(w_scd2))
# NULL valid_to = current record
sal_scd2.select("emp_id", "salary_after", "effective_date", "valid_to") \
        .filter(F.col("emp_id") == 5).show()
# Note: emp 5 has 2 rows with same effective_date (dup) — valid_to is non-deterministic for those


# ── Section 4 ── Point-in-time query ("what was salary on 2021-01-01?")
query_date = "2021-01-01"
sal_scd2.filter(
    (F.col("emp_id") == 5) &
    (F.col("effective_date") <= F.lit(query_date)) &
    (F.coalesce(F.col("valid_to"), F.lit("9999-12-31")) > F.lit(query_date))
).select("emp_id", "salary_after", "effective_date", "valid_to").show()
# Expected: 132000.0 (from 2020-01-01 raise, before 2022-04-01 raise)


# ── Section 5 ── Current record per employee (most recent row)
w_latest = Window.partitionBy("emp_id").orderBy(
    F.col("effective_date").desc(), F.col("hist_id").desc()
)
current_salary = sal.withColumn("rn", F.row_number().over(w_latest)) \
                    .filter(F.col("rn") == 1).drop("rn")
current_salary.select("emp_id", "salary_after", "effective_date").orderBy("emp_id").show(10)
# Comment: using hist_id desc as tiebreaker handles the emp 5 duplicate on same effective_date


# ── Section 6 ── Insert new Type 2 row (new salary change for emp 9)
from datetime import date
new_row = spark.createDataFrame(
    [(34, 9, 120000.0, 128000.0, date(2024, 7, 1), "Annual review 2024", 2)],
    schema="hist_id INT, emp_id INT, salary_before DOUBLE, salary_after DOUBLE, "
           "effective_date DATE, change_reason STRING, changed_by INT"
)
sal_updated = sal.union(new_row)
sal_updated.filter(F.col("emp_id") == 9).orderBy("effective_date").show()


# ── Section 7 ── SCD2 audit: raises per employee, avg gap between raises
w_sal = Window.partitionBy("emp_id").orderBy("effective_date")
sal.withColumn("next_date", F.lead("effective_date", 1).over(w_sal)) \
   .withColumn("days_to_next_raise", F.datediff("next_date", "effective_date")) \
   .groupBy("emp_id") \
   .agg(
       F.count("*").alias("num_changes"),
       F.avg("days_to_next_raise").alias("avg_days_between_raises"),
       F.sum(F.when(F.col("salary_before").isNull(), 0).otherwise(1)).alias("num_raises"),
   ).show()


stop_and_wait(spark)